# Stage 2: Safety Score Prediction (only with 200k)

This notebook predicts the perceived cycling safety class of a street-level image from visual features extracted by a computer vision pipeline. The target variable is derived from TrueSkill rankings produced in Stage 1.

**Pipeline:**
1. Install dependencies
2. Imports
3. Configuration
4. Load datasets
5. Composite feature construction
6. Feature cleaning (9 steps)
7. Feature family definitions
8. Model definitions
9. Shared functions
10. Tree hyperparameter search
11. Model runner functions
12. Plotting functions
13. Main CV loop
14. Results
15. Ablation B1: leave-one-group-out

**Datasets:**
- Visual features: segmentation coverage, centroid, perimeter, grid, object detection, image statistics
- GPT TrueSkill ranking: based on 225k combined synthetic votes (25k validated + 100k x 2 extended)
- Human TrueSkill ranking: based on 19,729 filtered crowdsourced votes

**Target:** 3-class safety label (Low / Medium / High) from quantile binning of TrueSkill mu

**Models:** Deeper MLP (256->128->64->32), XGBoost, LightGBM, CatBoost

**Evaluation:** 3 independent partitions x 5-fold location-level GroupKFold = 15 runs per model per dataset. All four headings from the same street location always stay together in the same fold to prevent location leakage.

**Ablation B1:** Leave-one-group-out across 10 feature families using the best model per dataset.

## 1. Install dependencies

In [ ]:
import sys, subprocess
print('Installing dependencies...')
for pkg in ['xgboost', 'lightgbm', 'catboost']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Done.')

## 2. Imports

In [ ]:
print('Loading libraries...')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import re
import sklearn
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, make_scorer
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
print('Imported.')

## 3. Configuration

**Edit `KAGGLE_INPUT` before running.** Update the dataset slug to match the Kaggle dataset where the three input files are uploaded.

In [ ]:
# ============================================================
# PATHS
# ============================================================
KAGGLE_INPUT    = '/kaggle/input/datasets/lsestudent/insumos2'
FEATURES_PATH   = f'{KAGGLE_INPUT}/processed_features_merged_02_06_26.csv'
GPT_RANK_PATH   = f'{KAGGLE_INPUT}/gpt_trueskill_rankings_200k_20_06_26.csv'
HUMAN_RANK_PATH = f'{KAGGLE_INPUT}/human_trueskill_rankings_final_20_06_26.csv'

# ============================================================
# SIGMA WEIGHTING
# 'none'      : no weighting (balanced class weights only)
# '1/sigma'   : inverse sigma
# '1/sigma^2' : inverse sigma squared (recommended)
# ============================================================
SIGMA_WEIGHTING = '1/sigma^2'

# ============================================================
# PEARSON CORRELATION THRESHOLD
# Features with |r| <= threshold are dropped before modelling.
# Computed against raw continuous mu, not the binned target.
# ============================================================
PEARSON_THRESHOLD_GPT   = 0.03
PEARSON_THRESHOLD_HUMAN = 0.03

# ============================================================
# CV SETTINGS
# ============================================================
NUM_BINS              = 3
BINNING_STRATEGY      = 'qcut'
BIN_LABELS            = ['Low', 'Medium', 'High']
N_SPLITS              = 5
N_PARTITIONS          = 3
VAL_SIZE_WITHIN_TRAIN = 0.15

# MLP training
EPOCHS     = 150
BATCH_SIZE = 64
PATIENCE   = 20

# Tree hyperparameter search
N_ITER_SEARCH = 12
CV_SEARCH     = 3

# Area normalisation
IMAGE_PIXELS = 640 * 640

# Style
FONT_FAMILY    = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
DATASET_COLORS = {'GPT': '#2171B5', 'Human': '#1B7837'}

print('Configuration loaded.')
print('=' * 55)
print(f'  Features path    : {FEATURES_PATH}')
print(f'  GPT ranking      : {GPT_RANK_PATH}')
print(f'  Human ranking    : {HUMAN_RANK_PATH}')
print(f'  Bins             : {NUM_BINS} ({", ".join(BIN_LABELS)})')
print(f'  Binning strategy : {BINNING_STRATEGY}')
print(f'  Pearson GPT      : |r| > {PEARSON_THRESHOLD_GPT}')
print(f'  Pearson Human    : |r| > {PEARSON_THRESHOLD_HUMAN}')
print(f'  Sigma weighting  : {SIGMA_WEIGHTING}')
print(f'  CV               : {N_PARTITIONS} x {N_SPLITS} = {N_PARTITIONS*N_SPLITS} evals per model')
print(f'  Random baseline  : {1/NUM_BINS*100:.2f}%')

## 4. Load datasets

In [ ]:
print('Loading datasets...')
raw_features_df = pd.read_csv(FEATURES_PATH)
gpt_df          = pd.read_csv(GPT_RANK_PATH)
human_df        = pd.read_csv(HUMAN_RANK_PATH)

gpt_df['image_name']   = gpt_df['image_name'].apply(lambda x: __import__('os').path.basename(str(x).strip()))
human_df['image_name'] = human_df['image_name'].apply(lambda x: __import__('os').path.basename(str(x).strip()))

print('=' * 55)
print(f'  Features : {raw_features_df.shape[0]:,} rows x {raw_features_df.shape[1]:,} columns')
print(f'  GPT      : {gpt_df.shape[0]:,} rows | columns: {list(gpt_df.columns)}')
print(f'  Human    : {human_df.shape[0]:,} rows | columns: {list(human_df.columns)}')
print(f'  GPT   sigma range : [{gpt_df["gpt_sigma"].min():.4f}, {gpt_df["gpt_sigma"].max():.4f}]')
print(f'  Human sigma range : [{human_df["human_sigma"].min():.4f}, {human_df["human_sigma"].max():.4f}]')

## 5. Composite feature construction

Composite features are constructed from raw segmentation coverage columns before cleaning. This ensures cleaning steps 8 and 9 can also remove any zero-variance or near-zero composites.

Each composite was validated against TrueSkill mu via Pearson correlation. Only composites where the combined signal exceeded the best individual constituent are included, except cycling infrastructure composites which are included for completeness.

| Composite | Type | Justification |
|---|---|---|
| comp_cycling_infra | coverage | near-zero r but included for completeness |
| comp_motor_load | coverage | avg|r|=0.21, beats seg_Car alone (0.18) |
| comp_separation | coverage | avg|r|=0.11, individual components weak individually |
| comp_natural_openness | coverage | avg|r|=0.21, beats Vegetation and Terrain individually |
| comp_commercial_density | coverage | avg|r|=0.12, Billboard alone was 0.10 |
| comp_built_density | coverage | avg|r|=0.21, beats seg_Building alone |
| comp_has_* | binary | presence flags for each coverage composite |
| comp_nature_vs_built | ratio | natural openness / built density |
| comp_terrain_vs_motor | ratio | terrain+vegetation / motor load |

In [ ]:
def safe_sum(df_, cols):
    present = [c for c in cols if c in df_.columns]
    missing = [c for c in cols if c not in df_.columns]
    if missing:
        print(f'  WARNING: missing columns skipped: {missing}')
    return df_[present].fillna(0).sum(axis=1) if present else pd.Series(0, index=df_.index)


df_with_composites = raw_features_df.copy()

df_with_composites['comp_cycling_infra'] = safe_sum(df_with_composites, [
    'seg_Bike Lane', 'seg_Bike Rack', 'seg_Bicycle',
    'seg_Bicyclist', 'seg_Other Rider', 'seg_Wheeled Slow'
])
df_with_composites['comp_motor_load'] = safe_sum(df_with_composites, [
    'seg_Car', 'seg_Truck', 'seg_Bus', 'seg_Motorcycle', 'seg_Other Vehicle'
])
df_with_composites['comp_separation'] = safe_sum(df_with_composites, [
    'seg_Barrier', 'seg_Guard Rail', 'seg_Fence', 'seg_Wall', 'seg_Tunnel', 'seg_Bridge'
])
df_with_composites['comp_natural_openness'] = safe_sum(df_with_composites, [
    'seg_Vegetation', 'seg_Terrain', 'seg_Mountain', 'seg_Water'
])
df_with_composites['comp_commercial_density'] = safe_sum(df_with_composites, [
    'seg_Billboard', 'seg_Street Light', 'seg_Traffic Sign (Front)', 'seg_CCTV Camera'
])
df_with_composites['comp_built_density'] = safe_sum(df_with_composites, [
    'seg_Building', 'seg_Wall', 'seg_Bridge'
])

for suffix in ['cycling_infra', 'motor_load', 'separation',
               'natural_openness', 'commercial_density', 'built_density']:
    df_with_composites[f'comp_has_{suffix}'] = (
        df_with_composites[f'comp_{suffix}'] > 0
    ).astype(int)

df_with_composites['comp_nature_vs_built'] = (
    df_with_composites['comp_natural_openness'] /
    (df_with_composites['comp_built_density'] + 0.001)
)
df_with_composites['comp_terrain_vs_motor'] = (
    safe_sum(df_with_composites, ['seg_Terrain', 'seg_Vegetation']) /
    (df_with_composites['comp_motor_load'] + 0.001)
)

comp_cols = [c for c in df_with_composites.columns if c.startswith('comp_')]
print(f'Composite features added : {len(comp_cols)}')
print(f'Shape before cleaning    : {df_with_composites.shape[0]:,} rows x {df_with_composites.shape[1]:,} columns')
print()
print(f'{"Composite":<35} {"Type":<8} {"% non-zero":>12} {"Mean":>8}')
print('-' * 70)
for c in comp_cols:
    s      = df_with_composites[c].replace([np.inf, -np.inf], np.nan).fillna(0)
    pct_nz = (s > 0).mean() * 100
    ctype  = 'binary' if 'has_' in c else 'ratio' if 'vs' in c else 'coverage'
    print(f'{c:<35} {ctype:<8} {pct_nz:>11.1f}% {s.mean():>8.4f}')

## 6. Feature cleaning

Nine sequential steps applied to the feature matrix including composite columns.

- **Step 1**: Remove unnamed index columns (artefacts of CSV export)
- **Step 2**: Extract point ID from image name for location-level GroupKFold
- **Step 3**: NaN imputation — centroid and perimeter NaNs kept (absence differs from position 0); coverage, grid and composite NaNs filled with 0
- **Step 4**: Remove all-zero rows (failed Street View requests); composites excluded
- **Step 5**: Remove exact duplicate rows; composites excluded from duplicate check
- **Step 6**: Remove known sky-like duplicates (points 2285 and 2790)
- **Step 7**: Normalise object area features by image pixel count; guarded against double-normalisation
- **Step 8**: Remove zero-variance columns
- **Step 9**: Remove near-zero columns (less than 0.1% non-zero values)

In [ ]:
df            = df_with_composites.copy()
initial_shape = df.shape
_grid_re      = re.compile(r'_\d+_\d+$')
_all_raw      = [c for c in df.columns if c != 'image_name']
spatial_cols  = (
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_cx')] +
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_cy')] +
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_perimeter')]
)
area_cols    = [c for c in _all_raw if c.startswith('obj_') and '_area_' in c]
non_spatial  = [c for c in _all_raw if c not in spatial_cols]

# Step 1: remove unnamed columns
unnamed = [c for c in df.columns if str(c).startswith('Unnamed') or not str(c).strip()]
df      = df.drop(columns=unnamed) if unnamed else df
print(f'Step 1 — unnamed columns removed              : {len(unnamed)}')

# Step 2: extract point_id
df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')
n_locs = df['point_id'].nunique()
print(f'Step 2 — point_id extracted                   : {n_locs:,} unique locations')

# Step 3: selective NaN fill
non_spatial_present = [c for c in non_spatial if c in df.columns]
nan_non_spatial = int(df[non_spatial_present].isna().sum().sum())
nan_spatial     = int(df[spatial_cols].isna().sum().sum()) if spatial_cols else 0
df[non_spatial_present] = df[non_spatial_present].fillna(0)
print(f'Step 3 — coverage/grid NaNs filled with 0     : {nan_non_spatial:,}')
print(f'         centroid/perimeter NaNs kept as NaN  : {nan_spatial:,}')

# Step 4: remove invalid all-zero rows
check_cols    = [c for c in non_spatial_present if not c.startswith('comp_')]
all_zero_mask = (df[check_cols] == 0).all(axis=1)
invalid_imgs  = df.loc[all_zero_mask, 'image_name'].tolist()
df            = df[~all_zero_mask].copy()
print(f'Step 4 — invalid all-zero rows removed        : {len(invalid_imgs)}')
for img in invalid_imgs:
    print(f'         {img}')

# Step 5: remove exact duplicate rows
dup_check    = [c for c in non_spatial_present
                if c in df.columns and not c.startswith('comp_')]
dup_mask     = df[dup_check].duplicated(keep='first')
removed_dups = df.loc[dup_mask, 'image_name'].tolist()
df           = df[~dup_mask].copy()
print(f'Step 5 — exact duplicate rows removed         : {len(removed_dups)}')
for img in removed_dups:
    print(f'         {img}')

# Step 6: remove known sky-like duplicates (points 2285 and 2790)
sky_dups   = [
    '2285_0.jpg', '2285_90.jpg', '2285_180.jpg', '2285_270.jpg',
    '2790_0.jpg', '2790_90.jpg', '2790_180.jpg', '2790_270.jpg'
]
before_sky = len(df)
df         = df[~df['image_name'].isin(sky_dups)].copy()
print(f'Step 6 — sky-like duplicates removed          : {before_sky - len(df)}')

# Step 7: normalise object area features by image pixel count
area_present    = [c for c in area_cols if c in df.columns]
area_max_before = df[area_present].max().max() if area_present else 0
if area_max_before > 1.0:
    df[area_present] = df[area_present] / IMAGE_PIXELS
    area_max_after   = df[area_present].max().max() if area_present else 0
    print(f'Step 7 — object area normalised               : {len(area_present)} columns')
    print(f'         max before: {area_max_before:,.1f}  ->  max after: {area_max_after:.6f}')
else:
    print(f'Step 7 — object area already normalised (max={area_max_before:.4f}) — skipped')

# Step 8: remove zero-variance columns
feat_cols     = [c for c in df.columns if c not in ('image_name', 'point_id')]
n_unique      = df[feat_cols].nunique(dropna=False)
zero_var_cols = n_unique[n_unique <= 1].index.tolist()
df            = df.drop(columns=zero_var_cols)
print(f'Step 8 — zero-variance columns removed        : {len(zero_var_cols)}')

# Step 9: remove near-zero columns (< 0.1% non-zero)
feat_cols    = [c for c in df.columns if c not in ('image_name', 'point_id')]
nonzero_frac = df[feat_cols].apply(lambda s: (s != 0).mean())
sparse_cols  = nonzero_frac[nonzero_frac < 0.001].index.tolist()
df           = df.drop(columns=sparse_cols)
print(f'Step 9 — near-zero columns removed            : {len(sparse_cols)} (< 0.1% non-zero)')

feat_cols_final = [c for c in df.columns if c not in ('image_name', 'point_id')]
features_df     = df.copy()
print()
print(f'Before cleaning : {initial_shape[0]:,} rows x {initial_shape[1]:,} columns')
print(f'After cleaning  : {len(features_df):,} rows x {len(feat_cols_final):,} features')
print(f'Locations       : {features_df["point_id"].nunique():,}')

## 7. Feature family definitions

In [ ]:
def get_family(name):
    if name.startswith('comp_'):          return 'Composite'
    if name.startswith('seg_'):
        if _grid_re.search(name):         return 'Seg grid'
        elif name.endswith('_cx'):        return 'Seg centroid X'
        elif name.endswith('_cy'):        return 'Seg centroid Y'
        elif name.endswith('_perimeter'): return 'Seg perimeter'
        else:                             return 'Seg coverage'
    elif name.startswith('obj_'):
        if '_count_' in name:             return 'Obj count'
        elif '_area_' in name:            return 'Obj area'
        else:                             return 'Obj probability'
    else:
        return 'Image statistics'


FAMILY_COLORS = {
    'Composite'       : '#8B4513',
    'Seg coverage'    : '#2171B5',
    'Seg centroid X'  : '#4292C6',
    'Seg centroid Y'  : '#6BAED6',
    'Seg perimeter'   : '#9ECAE1',
    'Seg grid'        : '#08519C',
    'Obj probability' : '#E8743B',
    'Obj count'       : '#F4A460',
    'Obj area'        : '#D2691E',
    'Image statistics': '#1B7837',
}
ALL_FAMILIES = list(FAMILY_COLORS.keys())

feat_cols_list = [c for c in features_df.columns if c not in ('image_name', 'point_id')]
family_map     = {c: get_family(c) for c in feat_cols_list}
fam_counts     = pd.Series(family_map).value_counts().reindex(ALL_FAMILIES, fill_value=0)

print('Feature families after cleaning:')
print(f'  {"Family":<22} {"Count":>6}  {"NaN cols":>10}')
print('  ' + '-' * 42)
for fam, cnt in fam_counts.items():
    cols_f   = [c for c in feat_cols_list if get_family(c) == fam]
    nan_cols = sum(features_df[c].isna().any() for c in cols_f)
    print(f'  {fam:<22} {cnt:>6}  {nan_cols:>10}')
print('  ' + '-' * 42)
print(f'  {"TOTAL":<22} {fam_counts.sum():>6}')

## 8. Model definitions

**Deeper MLP**: 4 hidden layers (256->128->64->32) with BatchNorm and Dropout. Centroid and perimeter NaNs are imputed with 0.5 before scaling (0.5 represents the image centre, the most neutral value). Imputation is done inside each CV fold to prevent leakage.

**XGBoost, LightGBM, CatBoost**: gradient boosting trees. All three handle NaN natively so no imputation is needed.

In [ ]:
def create_deeper_mlp(input_dim, num_classes):
    model = Sequential([
        Dense(256, input_shape=(input_dim,)),
        BatchNormalization(), Activation('relu'), Dropout(0.3),
        Dense(128),
        BatchNormalization(), Activation('relu'), Dropout(0.3),
        Dense(64),
        BatchNormalization(), Activation('relu'), Dropout(0.3),
        Dense(32),
        Activation('relu'), Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ], name='Deeper_MLP')
    model.compile(
        optimizer=Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


MODEL_ORDER = ['Deeper_MLP', 'XGBoost', 'LightGBM', 'CatBoost']

print('Models defined.')
print('  Deeper_MLP : 256->128->64->32->output')
print('               Dense -> BatchNorm -> ReLU -> Dropout')
print('  XGBoost    : gradient boosting, level-wise trees')
print('  LightGBM   : gradient boosting, leaf-wise trees')
print('  CatBoost   : gradient boosting, symmetric oblivious trees')

## 9. Shared functions

In [ ]:
def bin_mu(series):
    return pd.qcut(series, q=NUM_BINS, labels=False, duplicates='drop')


def select_features_pearson(features_df_in, merged_df, mu_col, threshold):
    all_feats = [c for c in features_df_in.columns
                 if c not in ('image_name', 'point_id') and c in merged_df.columns]
    mu_values = merged_df[mu_col].values
    corr_vals = {}
    for feat in all_feats:
        x = merged_df[feat].replace([np.inf, -np.inf], np.nan).fillna(0).values
        corr_vals[feat] = 0.0 if x.std() == 0 else float(pearsonr(x, mu_values)[0])
    corr_df = (
        pd.DataFrame({'feature': list(corr_vals.keys()),
                      'pearson_r': list(corr_vals.values())})
          .assign(abs_r=lambda d: d['pearson_r'].abs())
          .sort_values('abs_r', ascending=False)
          .reset_index(drop=True)
    )
    selected = corr_df[corr_df['abs_r'] > threshold]['feature'].tolist()
    print(f'  Pearson threshold : |r| > {threshold}')
    print(f'  Features kept     : {len(selected)} / {len(all_feats)}')
    print(f'  Top 10 by |r|:')
    for _, row in corr_df.head(10).iterrows():
        print(f'    {row["feature"]:<45s} r={row["pearson_r"]:+.4f}')
    return selected, corr_df


def prep_data(features_df_in, votes_df, mu_col, sigma_col, threshold):
    merged = votes_df.merge(features_df_in, on='image_name', how='inner')
    merged['mu_bin'] = bin_mu(merged[mu_col])
    print('  Computing Pearson correlations with raw mu...')
    selected, corr_df = select_features_pearson(
        features_df_in, merged, mu_col, threshold
    )
    zv = [f for f in selected if merged[f].nunique() <= 1]
    if zv:
        selected = [f for f in selected if f not in zv]
        print(f'  Zero-variance post-merge dropped: {len(zv)}')
    return merged, selected, corr_df


def get_sample_weights(sigma_values, y_values, class_weights):
    if SIGMA_WEIGHTING == 'none':
        sw = np.ones(len(sigma_values))
    elif SIGMA_WEIGHTING == '1/sigma':
        sw = 1.0 / np.maximum(sigma_values, 1e-6)
    elif SIGMA_WEIGHTING == '1/sigma^2':
        sw = 1.0 / np.maximum(sigma_values ** 2, 1e-12)
    else:
        raise ValueError(f'Unknown SIGMA_WEIGHTING: {SIGMA_WEIGHTING}')
    sw       = sw / sw.mean()
    cw       = np.array([class_weights[yi] for yi in y_values])
    combined = sw * cw
    return combined / combined.mean()


def preprocess_for_mlp(X_tr, X_val, X_te):
    # centroid/perimeter NaNs imputed with 0.5 (image centre = neutral position)
    # imputation fitted on training fold only to prevent leakage
    imp    = SimpleImputer(strategy='constant', fill_value=0.5)
    scaler = StandardScaler()
    X_tr_i  = imp.fit_transform(X_tr)
    X_val_i = imp.transform(X_val)
    X_te_i  = imp.transform(X_te)
    X_tr_s  = scaler.fit_transform(X_tr_i)
    X_val_s = scaler.transform(X_val_i)
    X_te_s  = scaler.transform(X_te_i)
    return X_tr_s, X_val_s, X_te_s


print('Shared functions defined.')

## 10. Tree hyperparameter search

Run once per dataset on a 20% location-level hold-out using `GroupShuffleSplit`. This ensures no street location appears in both the search training set and the search test set, consistent with the location-level grouping used in the main CV loop. Best parameters are reused across all 15 CV runs for a fair comparison. NaN values are filled with 0 for the search only as tree models handle NaN natively in the main CV loop.

In [ ]:
def tune_trees(X, y, sigma, groups, label):
    print(f'  Tuning tree models for {label}...')
    # location-level split: all headings from the same point_id stay together
    # GroupShuffleSplit ensures no location leakage in the hold-out
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    tr_idx, _ = next(gss.split(X, y, groups=groups))
    X_s   = X.iloc[tr_idx]
    y_s   = y.iloc[tr_idx]
    sig_s = sigma.iloc[tr_idx]

    cw_arr = compute_class_weight('balanced', classes=np.unique(y_s), y=y_s)
    cw     = {i: w for i, w in enumerate(cw_arr)}
    sw     = get_sample_weights(sig_s.values, y_s.values, cw)
    scorer = make_scorer(balanced_accuracy_score)

    xgb_search = RandomizedSearchCV(
        XGBClassifier(subsample=0.8, colsample_bytree=0.8,
                      eval_metric='mlogloss', random_state=SEED, verbosity=0),
        {'n_estimators': [200, 300, 400], 'max_depth': [4, 6, 8],
         'learning_rate': [0.05, 0.10]},
        n_iter=N_ITER_SEARCH, scoring=scorer,
        cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
    )
    xgb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
    best_xgb = xgb_search.best_params_
    print(f'  XGBoost  best: {best_xgb}  BA={xgb_search.best_score_:.4f}')

    lgb_search = RandomizedSearchCV(
        LGBMClassifier(subsample=0.8, colsample_bytree=0.8,
                       random_state=SEED, verbosity=-1),
        {'n_estimators': [200, 300, 400], 'max_depth': [4, 6, 8, -1],
         'learning_rate': [0.05, 0.10], 'num_leaves': [31, 63, 127]},
        n_iter=N_ITER_SEARCH, scoring=scorer,
        cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
    )
    lgb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
    best_lgb = lgb_search.best_params_
    print(f'  LightGBM best: {best_lgb}  BA={lgb_search.best_score_:.4f}')

    cat_search = RandomizedSearchCV(
        CatBoostClassifier(random_seed=SEED, verbose=0),
        {'iterations': [200, 300, 400], 'depth': [4, 6, 8],
         'learning_rate': [0.05, 0.10]},
        n_iter=N_ITER_SEARCH, scoring=scorer,
        cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
    )
    cat_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
    best_cat = cat_search.best_params_
    print(f'  CatBoost best: {best_cat}  BA={cat_search.best_score_:.4f}')

    return {'xgb': best_xgb, 'lgb': best_lgb, 'cat': best_cat}


print('=' * 70)
print('HYPERPARAMETER SEARCH')
print('=' * 70)
print(f'20% location-level hold-out | {N_ITER_SEARCH} iters each | {CV_SEARCH}-fold | sigma weights')
print()

feat_cols_all = [c for c in features_df.columns
                 if c not in ('image_name', 'point_id')]
tree_params   = {}

for ds_label, df_, mu_col, sigma_col, thr in [
    ('GPT',   gpt_df,   'gpt_mu',   'gpt_sigma',   PEARSON_THRESHOLD_GPT),
    ('Human', human_df, 'human_mu', 'human_sigma', PEARSON_THRESHOLD_HUMAN),
]:
    merged_s           = df_.merge(features_df, on='image_name', how='inner')
    merged_s['mu_bin'] = bin_mu(merged_s[mu_col])
    X_s = merged_s[feat_cols_all]
    y_s = merged_s['mu_bin']
    s_s = merged_s[sigma_col]
    g_s = merged_s['point_id']
    tree_params[ds_label] = tune_trees(X_s, y_s, s_s, g_s, ds_label)
    print()

print('Search complete.')

## 11. Model runner functions

Location-level splitting: `StratifiedGroupKFold` with `groups=point_id`. All four headings from the same street point stay in the same fold. The MLP stores mean absolute first-layer weights as a proxy for feature importance.

In [ ]:
def run_all_models(X, y, sigma, groups, label, tree_params_ds):
    results     = []
    importances = {}
    feat_names  = list(X.columns)
    num_classes = NUM_BINS

    print(f'  Deeper_MLP ({N_PARTITIONS}x{N_SPLITS} folds)...')
    all_scores, all_cms, all_wmags = [], [], []
    for part in range(N_PARTITIONS):
        sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
        split_iter = list(sgkf.split(X, y, groups=groups))
        rng        = np.random.RandomState(SEED + part * 100)
        rng.shuffle(split_iter)
        for fold, (tr_idx, te_idx) in enumerate(split_iter):
            X_trf  = X.iloc[tr_idx];     y_trf  = y.iloc[tr_idx]
            s_trf  = sigma.iloc[tr_idx]; X_te   = X.iloc[te_idx]
            y_te   = y.iloc[te_idx]
            tr_i, val_i = train_test_split(
                np.arange(len(X_trf)), test_size=VAL_SIZE_WITHIN_TRAIN,
                stratify=y_trf.values,
                random_state=SEED + part * 100 + fold
            )
            X_tr  = X_trf.iloc[tr_i];  y_tr  = y_trf.iloc[tr_i];  s_tr  = s_trf.iloc[tr_i]
            X_val = X_trf.iloc[val_i]; y_val = y_trf.iloc[val_i]; s_val = s_trf.iloc[val_i]
            X_tr_s, X_val_s, X_te_s = preprocess_for_mlp(X_tr, X_val, X_te)
            cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
            cw     = {i: w for i, w in enumerate(cw_arr)}
            sw_tr  = get_sample_weights(s_tr.values,  y_tr.values,  cw)
            sw_val = get_sample_weights(s_val.values, y_val.values, cw)
            model  = create_deeper_mlp(X_tr_s.shape[1], num_classes)
            model.fit(
                X_tr_s, y_tr,
                validation_data=(X_val_s, y_val, sw_val),
                epochs=EPOCHS, batch_size=BATCH_SIZE, sample_weight=sw_tr,
                callbacks=[
                    EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                  restore_best_weights=True, verbose=0),
                    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                     patience=10, min_lr=1e-6, verbose=0)
                ], verbose=0
            )
            y_pred = np.argmax(model.predict(X_te_s, verbose=0), axis=1)
            all_scores.append(balanced_accuracy_score(y_te, y_pred))
            all_cms.append(confusion_matrix(y_te, y_pred, labels=list(range(num_classes))))
            all_wmags.append(np.abs(model.layers[0].get_weights()[0]).mean(axis=1))
    mean_ba = np.mean(all_scores); std_ba = np.std(all_scores)
    results.append({'model': 'Deeper_MLP', 'type': 'MLP',
                    'bal_acc_mean': mean_ba, 'bal_acc_std': std_ba,
                    'bal_acc_min': np.min(all_scores), 'bal_acc_max': np.max(all_scores),
                    'confusion_matrix': np.mean(all_cms, axis=0)})
    importances['Deeper_MLP'] = (feat_names, np.mean(all_wmags, axis=0))
    print(f'    >> {mean_ba:.4f} +/- {std_ba:.4f}  [{np.min(all_scores):.4f}, {np.max(all_scores):.4f}]')

    tree_cfgs = [
        ('XGBoost',
         XGBClassifier(**tree_params_ds['xgb'], subsample=0.8, colsample_bytree=0.8,
                       eval_metric='mlogloss', random_state=SEED, verbosity=0)),
        ('LightGBM',
         LGBMClassifier(**tree_params_ds['lgb'], subsample=0.8, colsample_bytree=0.8,
                        random_state=SEED, verbosity=-1)),
        ('CatBoost',
         CatBoostClassifier(**tree_params_ds['cat'], random_seed=SEED, verbose=0)),
    ]
    for model_name, clf_template in tree_cfgs:
        print(f'  {model_name} ({N_PARTITIONS}x{N_SPLITS} folds)...')
        all_scores_t, all_cms_t, all_imp_t = [], [], []
        for part in range(N_PARTITIONS):
            sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
            split_iter = list(sgkf.split(X, y, groups=groups))
            rng        = np.random.RandomState(SEED + part * 100)
            rng.shuffle(split_iter)
            for tr_idx, te_idx in split_iter:
                X_tr = X.iloc[tr_idx]; y_tr = y.iloc[tr_idx]; s_tr = sigma.iloc[tr_idx]
                X_te = X.iloc[te_idx]; y_te = y.iloc[te_idx]
                cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
                cw     = {i: w for i, w in enumerate(cw_arr)}
                sw_tr  = get_sample_weights(s_tr.values, y_tr.values, cw)
                clf    = sklearn.base.clone(clf_template)
                clf.fit(X_tr, y_tr, sample_weight=sw_tr)
                y_pred = clf.predict(X_te)
                all_scores_t.append(balanced_accuracy_score(y_te, y_pred))
                all_cms_t.append(confusion_matrix(y_te, y_pred, labels=list(range(num_classes))))
                all_imp_t.append(clf.feature_importances_)
        mean_ba = np.mean(all_scores_t); std_ba = np.std(all_scores_t)
        key     = {'XGBoost': 'xgb', 'LightGBM': 'lgb', 'CatBoost': 'cat'}[model_name]
        results.append({'model': model_name, 'type': 'Tree',
                        'bal_acc_mean': mean_ba, 'bal_acc_std': std_ba,
                        'bal_acc_min': np.min(all_scores_t),
                        'bal_acc_max': np.max(all_scores_t),
                        'confusion_matrix': np.mean(all_cms_t, axis=0),
                        'best_params': tree_params_ds[key]})
        importances[model_name] = (feat_names, np.mean(all_imp_t, axis=0))
        print(f'    >> {mean_ba:.4f} +/- {std_ba:.4f}  [{np.min(all_scores_t):.4f}, {np.max(all_scores_t):.4f}]')

    return results, importances


def run_single_model(X, y, sigma, groups, model_name, model_obj):
    all_scores = []
    for part in range(N_PARTITIONS):
        sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
        split_iter = list(sgkf.split(X, y, groups=groups))
        rng        = np.random.RandomState(SEED + part * 100)
        rng.shuffle(split_iter)
        for fold, (tr_idx, te_idx) in enumerate(split_iter):
            X_trf = X.iloc[tr_idx]; y_trf = y.iloc[tr_idx]; s_trf = sigma.iloc[tr_idx]
            X_te  = X.iloc[te_idx]; y_te  = y.iloc[te_idx]
            cw_arr = compute_class_weight('balanced', classes=np.unique(y_trf), y=y_trf)
            cw     = {i: w for i, w in enumerate(cw_arr)}
            sw     = get_sample_weights(s_trf.values, y_trf.values, cw)
            if model_name == 'Deeper_MLP':
                tr_i, val_i = train_test_split(
                    np.arange(len(X_trf)), test_size=VAL_SIZE_WITHIN_TRAIN,
                    stratify=y_trf.values, random_state=SEED + part * 100 + fold
                )
                X_tr  = X_trf.iloc[tr_i]; y_tr  = y_trf.iloc[tr_i]; s_tr  = s_trf.iloc[tr_i]
                X_val = X_trf.iloc[val_i]; y_val = y_trf.iloc[val_i]; s_val = s_trf.iloc[val_i]
                X_tr_s, X_val_s, X_te_s = preprocess_for_mlp(X_tr, X_val, X_te)
                cw2    = {i: w for i, w in enumerate(
                    compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr))}
                sw_tr  = get_sample_weights(s_tr.values,  y_tr.values,  cw2)
                sw_val = get_sample_weights(s_val.values, y_val.values, cw2)
                m_mlp  = create_deeper_mlp(X_tr_s.shape[1], NUM_BINS)
                m_mlp.fit(
                    X_tr_s, y_tr,
                    validation_data=(X_val_s, y_val, sw_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, sample_weight=sw_tr,
                    callbacks=[
                        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                      restore_best_weights=True, verbose=0),
                        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=10, min_lr=1e-6, verbose=0)
                    ], verbose=0
                )
                y_pred = np.argmax(m_mlp.predict(X_te_s, verbose=0), axis=1)
            else:
                clf = sklearn.base.clone(model_obj)
                clf.fit(X_trf, y_trf, sample_weight=sw)
                y_pred = clf.predict(X_te)
            all_scores.append(balanced_accuracy_score(y_te, y_pred))
    return float(np.mean(all_scores)), float(np.std(all_scores))


print('Model runner functions defined.')

## 12. Plotting functions

In [ ]:
def print_results_table(results_dict):
    lookups = {ds: {r['model']: r for r in res} for ds, res in results_dict.items()}
    col_w   = 28
    header  = f"{'Model':<16} {'Type':<6}" + ''.join(f"{ds+' BA':>{col_w}}" for ds in results_dict)
    print(f'\n{header}')
    print('-' * (22 + col_w * len(results_dict)))
    for m in MODEL_ORDER:
        mtype = lookups[list(results_dict.keys())[0]][m]['type']
        row   = f'{m:<16} {mtype:<6}'
        for ds, lk in lookups.items():
            r    = lk[m]
            row += f"  {r['bal_acc_mean']:.4f} +/- {r['bal_acc_std']:.4f}"
        print(row)
    print('-' * (22 + col_w * len(results_dict)))
    baseline = f"{'Random baseline':<16} {'':6}"
    for _ in results_dict:
        baseline += f"  {1/NUM_BINS:>{col_w-2}.4f}"
    print(baseline)


def plot_results_bar(results_dict):
    lookups  = {ds: {r['model']: r for r in res} for ds, res in results_dict.items()}
    datasets = list(results_dict.keys())
    x        = np.arange(len(MODEL_ORDER))
    width    = 0.7 / len(datasets)
    fig, ax  = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    for i, ds in enumerate(datasets):
        means  = [lookups[ds][m]['bal_acc_mean'] for m in MODEL_ORDER]
        stds   = [lookups[ds][m]['bal_acc_std']  for m in MODEL_ORDER]
        offset = (i - (len(datasets)-1)/2) * width
        bars   = ax.bar(x + offset, means, width, yerr=stds, capsize=4,
                        alpha=0.85, color=DATASET_COLORS[ds], label=ds, edgecolor='black')
        for bar, mean, std in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.008,
                    f'{mean:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.axhline(1/NUM_BINS, color='red', linestyle='--', alpha=0.5, linewidth=1.5,
               label=f'Random baseline ({1/NUM_BINS*100:.1f}%)')
    ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, fontsize=10)
    ax.set_ylabel('Balanced Accuracy', fontsize=11)
    ax.set_title(
        f'Safety Score Prediction: {NUM_BINS} bins ({BINNING_STRATEGY}, {SIGMA_WEIGHTING} weighting)\n'
        f'Pearson |r| > 0.03 | Location-level GroupKFold | CV: {N_PARTITIONS}x{N_SPLITS}',
        fontsize=11, fontweight='bold'
    )
    ymax = max(lookups[ds][m]['bal_acc_mean'] + lookups[ds][m]['bal_acc_std']
               for ds in datasets for m in MODEL_ORDER)
    ax.set_ylim(0, ymax + 0.10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(loc='lower right', fontsize=10, frameon=True, framealpha=0.9)
    plt.tight_layout()
    plt.show()


def plot_confusion_and_recall(results_dict):
    datasets    = list(results_dict.keys())
    best_models = {ds: max(results_dict[ds], key=lambda r: r['bal_acc_mean'])
                   for ds in datasets}
    for ds, best in best_models.items():
        print(f'  Best {ds} model : {best["model"]} (BA={best["bal_acc_mean"]:.4f})')

    def _recall_table(cm, model_name, ds_label):
        cm        = np.array(cm)
        tp        = np.diag(cm)
        row_sums  = cm.sum(axis=1); col_sums = cm.sum(axis=0)
        recall    = np.where(row_sums > 0, tp/row_sums, 0.0)
        precision = np.where(col_sums > 0, tp/col_sums, 0.0)
        f1        = np.where((precision+recall) > 0,
                             2*precision*recall/(precision+recall), 0.0)
        print(f'\n  --- {ds_label} | {model_name} ---')
        print(f'  {"Bin":<4} {"Label":<14} {"Recall":>8} {"Precision":>10} {"F1":>8} {"Support":>9}')
        print('  ' + '-' * 58)
        for i in range(len(BIN_LABELS)):
            print(f'  {i:<4} {BIN_LABELS[i]:<14} {recall[i]:>8.3f} '
                  f'{precision[i]:>10.3f} {f1[i]:>8.3f} {int(row_sums[i]):>9,}')
        print('  ' + '-' * 58)
        print(f'  {"Mean":<20} {recall.mean():>8.3f} {precision.mean():>10.3f} {f1.mean():>8.3f}')

    for ds in datasets:
        _recall_table(best_models[ds]['confusion_matrix'], best_models[ds]['model'], ds)

    fig, axes = plt.subplots(1, len(datasets), figsize=(7*len(datasets), 6))
    fig.patch.set_facecolor('white')
    fig.suptitle(f'Confusion Matrices: best model per dataset ({NUM_BINS} bins, {BINNING_STRATEGY})\n'
                 'Row-normalised: diagonal = recall per class',
                 fontsize=13, fontweight='bold')
    if len(datasets) == 1: axes = [axes]
    for ax, ds in zip(axes, datasets):
        ax.set_facecolor('white')
        best  = best_models[ds]
        cm    = np.array(best['confusion_matrix'])
        row_s = cm.sum(axis=1, keepdims=True)
        cm_n  = np.where(row_s > 0, cm/row_s, 0.0)
        sns.heatmap(cm_n, annot=False, cmap='Blues', vmin=0, vmax=1, square=True,
                    ax=ax, xticklabels=BIN_LABELS, yticklabels=BIN_LABELS,
                    cbar_kws={'shrink': 0.8, 'label': 'Recall (row-normalised)'},
                    linewidths=0.5, linecolor='white')
        ax.set_facecolor('white')
        for i in range(cm_n.shape[0]):
            for j in range(cm_n.shape[1]):
                val = cm_n[i,j]; raw = int(round(cm[i,j]))
                ax.text(j+0.5, i+0.5, f'{val:.2f}\n({raw})',
                        ha='center', va='center', fontsize=8,
                        color='white' if val > 0.55 else 'black')
            ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                       edgecolor='red', linewidth=2.5))
        ax.set_title(f"{ds}: {best['model']}\nBA={best['bal_acc_mean']:.4f}",
                     fontsize=11, fontweight='bold', color='#111111')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_xticklabels(BIN_LABELS, rotation=45, ha='right')
        ax.set_yticklabels(BIN_LABELS, rotation=0)
    plt.tight_layout(rect=[0,0,1,0.93])
    plt.show()


def plot_feature_family_importance(importances_dict, dataset_label):
    model_names = list(importances_dict.keys())
    cat_imp     = {m: {f: 0.0 for f in ALL_FAMILIES} for m in model_names}
    for model_name, (feat_names, imp_vals) in importances_dict.items():
        imp_vals = np.array(imp_vals)
        total    = imp_vals.sum()
        if total > 0: imp_vals = imp_vals / total
        for feat, imp in zip(feat_names, imp_vals):
            cat_imp[model_name][get_family(feat)] += imp
    print(f'\n{dataset_label} — importance by feature family (normalised):')
    print(f'  {"Model":<16}' + ''.join(f'{f:>18}' for f in ALL_FAMILIES))
    print('  ' + '-' * (16 + 18*len(ALL_FAMILIES)))
    for m in model_names:
        vals = ''.join(f"{cat_imp[m][f]:>18.4f}" for f in ALL_FAMILIES)
        print(f'  {m:<16}{vals}')
    x = np.arange(len(model_names)); width = 0.08
    fig, ax = plt.subplots(figsize=(16, 5))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    for i, fam in enumerate(ALL_FAMILIES):
        vals   = [cat_imp[m][fam] for m in model_names]
        offset = (i - (len(ALL_FAMILIES)-1)/2) * width
        bars   = ax.bar(x+offset, vals, width, label=fam,
                        color=FAMILY_COLORS[fam], edgecolor='white', alpha=0.85)
        for bar, val in zip(bars, vals):
            if val > 0.01:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)
    ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=10)
    ax.set_ylabel('Summed normalised importance', fontsize=11)
    ax.set_title(f'Feature Family Importance — {dataset_label} ({NUM_BINS} bins)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, frameon=True, framealpha=0.9, ncol=5)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()


print('Plotting functions defined.')

## 13. Main CV loop

In [ ]:
print('=' * 70)
print(f'DATA PREPARATION — {NUM_BINS} BINS ({BINNING_STRATEGY})')
print('=' * 70)

merged_all = {}
feats_all  = {}
corr_all   = {}

for ds_label, votes_df, mu_col, sigma_col, thr in [
    ('GPT',   gpt_df,   'gpt_mu',   'gpt_sigma',   PEARSON_THRESHOLD_GPT),
    ('Human', human_df, 'human_mu', 'human_sigma', PEARSON_THRESHOLD_HUMAN),
]:
    print(f'\n{ds_label}:')
    m, f, c = prep_data(features_df, votes_df, mu_col, sigma_col, thr)
    merged_all[ds_label] = m
    feats_all[ds_label]  = f
    corr_all[ds_label]   = c
    bin_dist = m['mu_bin'].value_counts().sort_index()
    print(f'  Merged rows : {len(m):,} | Features selected: {len(f)}')
    print(f'  Locations   : {m["point_id"].nunique():,}')
    print(f'  Bin distribution: '
          + ' | '.join([f'Bin {b}={cnt} ({cnt/len(m)*100:.0f}%)'
                        for b, cnt in bin_dist.items()]))

results_all     = {}
importances_all = {}
sigma_cols      = {'GPT': 'gpt_sigma', 'Human': 'human_sigma'}

for ds_label in ['GPT', 'Human']:
    print('=' * 70)
    print(f'EXPERIMENTS — {ds_label} ({NUM_BINS} BINS)')
    print('=' * 70)
    m      = merged_all[ds_label]
    feats  = feats_all[ds_label]
    X      = m[feats].reset_index(drop=True)
    y      = m['mu_bin'].reset_index(drop=True)
    sigma  = m[sigma_cols[ds_label]].reset_index(drop=True)
    groups = m['point_id'].reset_index(drop=True)
    print(f'  Samples  : {len(X):,} | Features: {X.shape[1]}')
    print(f'  Locations: {groups.nunique():,} (GroupKFold — no location leakage)')
    print(f'  Random baseline: {1/NUM_BINS*100:.1f}%')
    print(f'  CV: {N_PARTITIONS} x {N_SPLITS} = {N_PARTITIONS*N_SPLITS} evaluations per model')
    print()
    res, imp = run_all_models(X, y, sigma, groups, ds_label, tree_params[ds_label])
    results_all[ds_label]     = res
    importances_all[ds_label] = imp
    print(f'\n{ds_label} experiments complete.')

lookups_all = {
    ds: {r['model']: r for r in res}
    for ds, res in results_all.items()
}
print('\nAll models complete.')

## 14. Results

In [ ]:
print('\n======================================================================')
print(f'RESULTS SUMMARY — {NUM_BINS} BINS ({BINNING_STRATEGY.upper()}, {SIGMA_WEIGHTING} WEIGHTING)')
print('======================================================================')
print_results_table(results_all)
print()
plot_results_bar(results_all)

print('======================================================================')
print('CONFUSION MATRICES + RECALL — BEST MODEL PER DATASET')
print('======================================================================')
plot_confusion_and_recall(results_all)

print('======================================================================')
print('FEATURE FAMILY IMPORTANCE')
print('======================================================================')
for ds_label in ['GPT', 'Human']:
    print(f'\n--- {ds_label} ---')
    plot_feature_family_importance(importances_all[ds_label], ds_label)

## 15. Ablation B1: leave-one-group-out

The best model per dataset is retrained ten times, each time excluding one complete feature family. Balanced accuracy drop is defined as full model BA minus ablated BA. A positive drop means removing that family reduced performance and the family is therefore contributing positively to prediction.

In [ ]:
def get_best_clf(ds_label):
    best_name = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])['model']
    tp = tree_params[ds_label]
    if best_name == 'Deeper_MLP':
        return best_name, None
    elif best_name == 'XGBoost':
        return best_name, XGBClassifier(
            **tp['xgb'], subsample=0.8, colsample_bytree=0.8,
            eval_metric='mlogloss', random_state=SEED, verbosity=0)
    elif best_name == 'LightGBM':
        return best_name, LGBMClassifier(
            **tp['lgb'], subsample=0.8, colsample_bytree=0.8,
            random_state=SEED, verbosity=-1)
    elif best_name == 'CatBoost':
        return best_name, CatBoostClassifier(
            **tp['cat'], random_seed=SEED, verbose=0)


print('=' * 70)
print('ABLATION B1 — LEAVE-ONE-GROUP-OUT')
print('=' * 70)
print(f'10 feature families | best model per dataset | {N_PARTITIONS}x{N_SPLITS} CV')
print('BA drop = Full BA minus Ablated BA  (positive = family helps)')
print()

ablation_b1 = {}

for ds_label in ['GPT', 'Human']:
    print(f'--- {ds_label} ---')
    m      = merged_all[ds_label]
    feats  = feats_all[ds_label]
    X_full = m[feats].reset_index(drop=True)
    y      = m['mu_bin'].reset_index(drop=True)
    sigma  = m[sigma_cols[ds_label]].reset_index(drop=True)
    groups = m['point_id'].reset_index(drop=True)
    best_name, clf_obj = get_best_clf(ds_label)
    full_ba = lookups_all[ds_label][best_name]['bal_acc_mean']
    print(f'  Best model : {best_name} | Full BA: {full_ba:.4f}')
    ds_res = {'full': full_ba, '_best_model': best_name}
    for fam in ALL_FAMILIES:
        kept      = [f for f in feats if get_family(f) != fam]
        n_dropped = len(feats) - len(kept)
        if len(kept) == 0:
            print(f'  Without {fam:<26}: no features remaining — skipped')
            ds_res[fam] = np.nan; continue
        X_abl = X_full[kept]
        mean_ba, std_ba = run_single_model(X_abl, y, sigma, groups, best_name, clf_obj)
        drop = full_ba - mean_ba
        ds_res[fam] = mean_ba
        print(f'  Without {fam:<26}: BA={mean_ba:.4f} +/- {std_ba:.4f} '
              f'| drop={drop:+.4f} | dropped={n_dropped}')
    ablation_b1[ds_label] = ds_res
    print()

print('=' * 70)
print('ABLATION B1 — SUMMARY TABLE AND CHART')
print('=' * 70)
print('Values shown: BA drop. Positive = removing this family hurts performance.')
print()
print(f'{"Feature family":<35} {"GPT drop":>12} {"Human drop":>12}')
print('-' * 60)
for fam in ALL_FAMILIES:
    gpt_drop   = ablation_b1['GPT']['full']   - ablation_b1['GPT'].get(fam, np.nan)
    human_drop = ablation_b1['Human']['full'] - ablation_b1['Human'].get(fam, np.nan)
    print(f'{fam:<35} {gpt_drop:>+12.4f} {human_drop:>+12.4f}')
print('-' * 60)
print(f'  Full BA GPT   : {ablation_b1["GPT"]["full"]:.4f}  Best: {ablation_b1["GPT"]["_best_model"]}')
print(f'  Full BA Human : {ablation_b1["Human"]["full"]:.4f}  Best: {ablation_b1["Human"]["_best_model"]}')

In [ ]:
x     = np.arange(len(ALL_FAMILIES))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for i, ds in enumerate(['GPT', 'Human']):
    drops  = [
        ablation_b1[ds]['full'] - ablation_b1[ds].get(f, np.nan)
        if not np.isnan(ablation_b1[ds].get(f, np.nan)) else 0.0
        for f in ALL_FAMILIES
    ]
    offset = (i - 0.5) * width
    bars   = ax.bar(x+offset, drops, width, label=ds,
                    color=DATASET_COLORS[ds], edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, drops):
        if abs(val) > 0.001:
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.001 if val >= 0 else bar.get_height()-0.003,
                    f'{val:+.3f}', ha='center', va='bottom', fontsize=8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(ALL_FAMILIES, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('BA drop (Full minus Ablated)', fontsize=11)
ax.set_title(
    f'Ablation B1: leave-one-group-out ({NUM_BINS} bins, {BINNING_STRATEGY})\n'
    'Positive = removing this family reduces balanced accuracy',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=10, frameon=True, framealpha=0.9, edgecolor='#cccccc')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
plt.tight_layout()
plt.show()